In [9]:
import sys                                           # 系统模块
import asyncio                                       # 异步 IO 模块
from dotenv import load_dotenv                       # 读取 .env
load_dotenv()                                        # 加载配置

# ---------------- Windows + pywin32 兼容补丁 ----------------
# 背景：mcp -> mcp.client.stdio -> mcp.os.win32.utilities 会 import pywintypes。
# pywintypes.py 位于 site-packages/win32/lib/，只有 pywin32.pth 被执行时才会进入 sys.path。
# 若 Jupyter 内核不是用本项目的 .venv 解释器启动（例如走了 PATH 上的其它 python），
# .pth 不会生效，就会报 ModuleNotFoundError: No module named 'pywintypes'。
# 这里手动补齐路径与 DLL 目录，保证两种情况都能跑。
if sys.platform == "win32":
    import os
    try:
        import pywintypes  # noqa: F401
    except Exception:                                # 通常是 ModuleNotFoundError
        try:
            import langchain_core
        except Exception:
            pass
        else:
            _sp = os.path.dirname(os.path.dirname(langchain_core.__file__))
            for _sub in ("win32", "win32/lib", "pythonwin"):
                _d = os.path.join(_sp, *_sub.split("/"))
                if os.path.isdir(_d) and _d not in sys.path:
                    sys.path.append(_d)
            _dll = os.path.join(_sp, "pywin32_system32")
            if os.path.isdir(_dll):
                os.add_dll_directory(_dll)
            try:
                import pywintypes  # noqa: F401
                print("[patch] pywin32 路径已补全，pywintypes 可正常导入")
            except Exception as _e:
                print("[patch] pywin32 仍不可用：", _e, "（请改用 .venv 解释器的内核）")
# ------------------------------------------------------------

if sys.platform == "win32":                          # 如果是 Windows 平台
    # Python 3.8+ 在 Windows 上默认就是 Proactor；3.14 起 set_event_loop_policy 已弃用
    if sys.version_info < (3, 14):
        if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    if "ipykernel" in sys.modules:                   # 如果运行在 Jupyter 内核里
        sys.stderr = sys.__stderr__                  # 恢复标准错误输出（避免某些显示问题）

from langchain_mcp_adapters.client import MultiServerMCPClient
client=MultiServerMCPClient({
    "time":{
        "transport":"stdio",
        "command":'uvx',
        "args":[
            "mcp-server-time",
            "--local-timezone=Asia/Shanghai",
        ]
    }
})
tools=await client.get_tools()
for tool in tools:
    print(tool.name)
from langchain_core.messages import HumanMessage
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model=init_chat_model(
    model='qwen3.5-plus',
    model_provider='openai',
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
agent=create_agent(
    model,
    tools
)
response=await agent.ainvoke(
    {
        "messages":HumanMessage("现在几点了")
    }
)
for message in response["messages"]:
    message.pretty_print()


[patch] pywin32 路径已补全，pywintypes 可正常导入
get_current_time
convert_time
get_time_ago
================================ Human Message =================================

现在几点了
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_dc387f595fc54cb2baf17e99)
 Call ID: call_dc387f595fc54cb2baf17e99
  Args:
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-14T20:34:24+08:00",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_23e20f28-c2cd-4d0f-988a-d8850f380897'}]
================================== Ai Message ==================================

现在是20:34:24（上海时间，2026年11月14日，星期六）。


In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient
client=MultiServerMCPClient(
    {
        "kiwi-com-flight-search":{
            "transport":"http",
            "url":'https://mcp.kiwi.com',
        }
    }
)
tools=await client.get_tools()
for tool in tools:
    print(tool.name)
model=init_chat_model(
    model="qwen3.5-plus",
    model_provider='openai',
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
agent=create_agent(
    model,
    tools,
    system_prompt="你是一名旅行智能助手，帮用户找到合适的航班"
)
response=await agent.ainvoke({
    "messages":[HumanMessage("帮我查询下周一深圳飞北京的航班")]
})
for message in response["messages"]:
    message.pretty_print()

ModuleNotFoundError: No module named 'pywintypes'